# 🔬 OpenAmadeus Companion — Kurisu Qwen3-TTS 1.7B Remote Worker

Este notebook executa o modelo **Kurisu Qwen3-TTS 1.7B** (`starrydark/Kurisu_Qwen3_TTS`) em GPUs em nuvem (**Google Colab Pro T4/L4/A100**), gerando um túnel HTTPS seguro via **Cloudflare Tunnel**.

### Vantagens:
- **0% de uso de VRAM local:** Ideal se você possui GPU local com menos de 8GB VRAM (como RTX 3050 6GB).
- **Alta Fidelidade:** Mantém prosódia, suspiros, hesitações e controle emocional *tsundere* autênticos de Makise Kurisu.
- **Conexão Instantânea:** Basta copiar o link `https://xxx.trycloudflare.com` e colar nas Configurações do OpenAmadeus Companion.

In [ ]:
# Passo 1: Verificar GPU disponível no Colab
!nvidia-smi

In [ ]:
# Passo 2: Instalar dependências necessárias
!pip install -q --upgrade pip
!pip install -q torch torchaudio --index-url https://download.pytorch.org/whl/cu124
!pip install -q qwen-tts soundfile fastapi uvicorn pydantic pycloudflared httpx

In [ ]:
# Passo 3: Iniciar Servidor Kurisu Qwen3-TTS com Túnel Cloudflare
import os
import io
import re
import time
import torch
import soundfile as sf
import numpy as np
from typing import Optional
from fastapi import FastAPI, HTTPException
from fastapi.middleware.cors import CORSMiddleware
from fastapi.responses import Response
from pydantic import BaseModel
import uvicorn
from pycloudflared import try_cloudflare
from qwen_tts import Qwen3TTSModel

print("=" * 65)
print("⏳ Carregando Kurisu Qwen3-TTS 1.7B na GPU...")
print("=" * 65)

device = "cuda:0" if torch.cuda.is_available() else "cpu"
dtype = torch.bfloat16 if torch.cuda.is_available() else torch.float32

model = Qwen3TTSModel.from_pretrained(
    "starrydark/Kurisu_Qwen3_TTS",
    device_map=device,
    dtype=dtype,
)

print("✅ Modelo residente na GPU com sucesso!")

app = FastAPI(title="OpenAmadeus Qwen3-TTS Colab Worker")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

class SynthesizeRequest(BaseModel):
    text: str
    language: Optional[str] = "Portuguese"
    instruct: Optional[str] = None
    temperature: Optional[float] = 0.75
    top_p: Optional[float] = 0.95
    repetition_penalty: Optional[float] = 1.05

@app.get("/")
def root():
    return {"system": "OpenAmadeus Qwen3-TTS Colab Worker", "status": "ready"}

@app.get("/health")
def health():
    vram = torch.cuda.memory_allocated() / (1024**2) if torch.cuda.is_available() else 0
    return {
        "status": "ready",
        "device": device,
        "cuda_available": torch.cuda.is_available(),
        "vram_allocated_mb": round(vram, 2),
        "model": "starrydark/Kurisu_Qwen3_TTS",
    }

@app.post("/synthesize")
async def synthesize(req: SynthesizeRequest):
    text = (req.text or "").strip()
    if not text:
        raise HTTPException(status_code=400, detail="Texto vazio.")

    t0 = time.perf_counter()
    with torch.inference_mode():
        wavs, sr = model.generate_custom_voice(
            text=text,
            speaker="kurisu",
            language=req.language or "Portuguese",
            instruct=req.instruct or "Fale em tom tsundere e expressivo, som seco e direto sem eco.",
            temperature=req.temperature or 0.75,
            top_p=req.top_p or 0.95,
            repetition_penalty=req.repetition_penalty or 1.05,
        )

    audio = wavs[0]
    if isinstance(audio, torch.Tensor):
        audio = audio.detach().cpu().numpy()
    if audio.ndim > 1:
        audio = audio.squeeze()

    buf = io.BytesIO()
    sf.write(buf, audio, sr, format="WAV", subtype="PCM_16")
    buf.seek(0)
    elapsed = round(time.perf_counter() - t0, 3)
    print(f"🎙️ Síntese: {len(text)} caracteres em {elapsed}s")
    return Response(content=buf.read(), media_type="audio/wav")

# Abrir Túnel Cloudflare na porta 8005
PORT = 8005
tunnel_res = try_cloudflare(port=PORT)
tunnel_url = tunnel_res.tunnel if hasattr(tunnel_res, 'tunnel') else str(tunnel_res)
print("\n" + "=" * 65)
print("  🌟 URL PÚBLICA DO CLOUDFLARE TUNNEL:")
print(f"  👉 {tunnel_url}")
print("=" * 65)
print("  Copie e cole este link em 'Configurações > Voz > URL Remota (Colab)' no OpenAmadeus Companion!\n")

config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="info")
server = uvicorn.Server(config)
await server.serve()